# Chapter_2_Group_Exercise_2_Regression_Analysis_and_Feature_Selection

Group Members:

1. Arya Shinde – Matriculation Number: 100006646
2. Mirang Bhandari – Matriculation Number: 100007049
3. Yash Annapure – Matriculation Number: 100006547
4. Anushka Sawant – Matriculation Number: 100006644

# Regression Analysis and Feature Selection

Perform regression analysis on a dataset with over 1000 data points and at least 30 features. Apply preprocessing, build models, evaluate performance, and enhance them using feature selection methods. Demonstrate creativity with additional techniques or insights.

### 1. Preprocessing
- 1. Handle missing values ​​and outliers.
- 2. Scale features using standardization or normalization.
- 3. Perform feature selection.
### 2. Build and Evaluate Models
- 1. Train a Linear Regression model.
- 2.  Evaluate using R² Score and RMSE.
- 3. Perform K-Fold Cross-Validation (eg, 5-fold).
### 3. Enhance the Model
- 1. Experiment with feature selection techniques and analyze their impact.
- 2. Visualize results (eg, feature importance, residual plots).
### 4. Creativity
- 1. Try advanced models (eg, Ridge, Lasso, Polynomial).
- 2. Create insightful visualizations or optimize hyperparameters.
- 3. Put the nice text justification on your shared colab

# Step 1: Source the dataset and handle missing values
The California Housing dataset is a real-world dataset derived from the 1990 U.S. Census, containing information about various housing districts across California. It includes 20,640 observations with features such as median income, housing age, total rooms, total bedrooms, population, households, and the categorical variable ocean_proximity, which indicates the district’s location relative to the ocean. The target variable, median_house_value, represents the median value of houses in each district and is used for regression tasks. 


In [2]:
import pandas as pd

cal_data = pd.read_csv("../Dataset/housing.csv")
cal_data.head(20)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY
5,-122.25,37.85,52.0,919.0,213.0,413.0,193.0,4.0368,269700.0,NEAR BAY
6,-122.25,37.84,52.0,2535.0,489.0,1094.0,514.0,3.6591,299200.0,NEAR BAY
7,-122.25,37.84,52.0,3104.0,687.0,1157.0,647.0,3.1200,241400.0,NEAR BAY
8,-122.26,37.84,42.0,2555.0,665.0,1206.0,595.0,2.0804,226700.0,NEAR BAY
9,-122.25,37.84,52.0,3549.0,707.0,1551.0,714.0,3.6912,261100.0,NEAR BAY


## Step 1.1: Check Missing Values and sum them up

In [3]:
cal_data.isnull().sum()

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

## Step 1.2: Handle Missing Values
Only 'total_bedrooms' contains missing values. So we will be using median imputation. Median imputation is used to handle missing values in the total_bedrooms feature, as it is robust to outliers and preserves the original distribution better than mean imputation.

In [4]:
cal_data = cal_data.fillna(cal_data['total_bedrooms'].median())

In [5]:
cal_data.isnull().sum()

longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
ocean_proximity       0
dtype: int64

## Step 1.3: Handle Outliers
The following columns have outliers: 
'median_house_value', 'median_income', 'total_rooms', 'population'. We will doing log transformation to handle these outliers since log transformation reduces skewness and minimizes the impact of extreme values while retaining all observations.


In [7]:
import numpy as np

cal_data['median_house_value'] = np.log1p(cal_data['median_house_value'])
cal_data['median_income'] = np.log1p(cal_data['median_income'])

# Step 2: Feature Scaling

## Step 2.1 Encode Categorical Variable
'ocean_proximity' is the categorical column here.

In [10]:
cal_data = pd.get_dummies(cal_data, columns=['ocean_proximity'], drop_first=True)


## Step 2.2: Standardization
Standardization ensures that all numerical features are on the same scale, which improves convergence and performance of regression models.

In [11]:
X = cal_data.drop('median_house_value', axis=1)
y = cal_data['median_house_value']


In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 3: Feature Selection

## Step 3.1: Correlation-Based Feature Selection

In [14]:
corr = cal_data.corr()['median_house_value'].abs()
selected_features = corr[corr > 0.1].index
df_corr_selected = cal_data[selected_features]

Features with low correlation to the target variable are unlikely to contribute significantly to predictive performance and can introduce noise.

## Step 3.2: Lasso (L1 Regularization)

In [15]:
from sklearn.linear_model import Lasso
from sklearn.feature_selection import SelectFromModel

lasso = Lasso(alpha=0.001)
lasso.fit(X_scaled, y)

model = SelectFromModel(lasso, prefit=True)
X_lasso_selected = model.transform(X_scaled)

Lasso regression performs embedded feature selection by shrinking less important feature coefficients to zero, resulting in a simpler and more interpretable model.

## Step 3.3: Recursive Feature Elimination (RFE)

In [16]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

rfe = RFE(estimator=LinearRegression(), n_features_to_select=15)
X_rfe = rfe.fit_transform(X_scaled, y)


C:\Users\Anushka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\feature_selection\_rfe.py:300: UserWarning: Found n_features_to_select=15 > n_features=12. There will be no feature selection and all features will be kept.
  warnings.warn(


RFE systematically eliminates less important features, improving model performance and reducing overfitting.